## Playbook 3 — runaway process pinning the CPU

**Symptom:** load average climbing, fans spinning, system unresponsive.

**1. Find the culprit:**

```bash
top      # P sorts by CPU (default);  htop is nicer
ps -eo pid,user,%cpu,comm --sort=-%cpu | head
```

**2. What is it actually doing?** Once you have a PID, drill in:

```bash
sudo strace -p 12345          # what syscalls is it making right now?
sudo strace -c -p 12345       # a summary count (Ctrl-C to stop)
cat /proc/12345/cmdline | tr '\0' ' '   # its full command line
```

**`strace`** is the star here — a process spinning in a tight syscall loop (say, reopening the same file forever) reveals the bug instantly.

**3. Control it — escalate gently:**

```bash
sudo renice +10 -p 12345      # lower priority (module 05) — don't kill, just yield
sudo kill -STOP 12345         # pause to inspect...
sudo kill -CONT 12345         # ...then resume
kill 12345; sleep 5; kill -9 12345   # SIGTERM, then SIGKILL if needed
```

**4. Can't kill it (state `D`)?** A process in **uninterruptible sleep** — blocked in the kernel on disk or NFS — **cannot be killed, not even with `-9`** (module 05). Your only options: wait for the I/O to complete, fix the underlying cause (e.g. a wedged NFS mount), or reboot.
